# FRM's true ceiling: what can you predict WITHOUT the question?

The teachers score 35.0% (attn x grad) and 31.1% (attention) **while reading the question**.
FRM never sees the question — it computes `r = FRM(gaze_tok, G)` and predicts the marginal
`E_q[imp | gaze, G]`. So 35% is a bound FRM is structurally forbidden from reaching, and the
21.7% gaze-geometry number is a floor that ignores the image entirely.

The real ceiling sits between them. This measures it by isolating what each information source buys:

| predictor | gaze | image | question |
|---|---|---|---|
| CTRL random / center | — | — | — |
| gaze blob (geometry) | yes | — | — |
| **token distinctiveness** | — | yes | — |
| **null-prompt attention** | — | yes | — |
| **gaze-token similarity** = *untrained FRM* | yes | yes | — |
| gaze blob x gaze-similarity | yes | yes | — |
| attention `imp_a` | — | yes | **yes** |
| attn x grad | — | yes | **yes** |

**Gaze-token similarity is literally the FRM architecture with identity projections**
(`s = (W_k G)(W_q gaze)/sqrt(d)` with `W_q = W_k = I`). It is what FRM scores *before any
training*, so it is a lower bound on a trained module and tells you whether the gaze token's
embedding carries usable information about which other tokens matter at all.

**Two tables.** The first uses the same candidate set as every earlier notebook (sinks removed).
The second **also removes the fovea** (gaze cell + its 1-ring), which is FRM's actual operating
regime per spec section 3.1 — the fovea is already rendered at high resolution by Stage 2a, so FRM
only ever picks from tokens outside it. **That second table is the one that matters.**

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib scipy
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math, glob, json, time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.stats import norm

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import rater_selection as RS
import visual_selection as VS

DATA_ROOT  = "/content/drive/MyDrive/wearvqa_gaze_only"
CACHE      = "/content/drive/MyDrive/wearvqa_faithfulness.pt"
SINKF      = "/content/drive/MyDrive/sink_mask_smolvlm2.pt"
MODEL_ID   = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
NULL_PROMPT = "Describe the image."      # content-free stand-in for a real question
N_PER_TYPE, GROUP = 2, 1

model, processor, device = S._load_smolvlm(MODEL_ID)
tokenizer = processor.tokenizer

def build_inputs(image, question, answer=""):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    full = processor(text=prompt + answer, images=[image], return_tensors="pt").to(device)
    only = processor(text=prompt, images=[image], return_tensors="pt")
    return full, int(only["input_ids"].shape[1])

@torch.no_grad()
def answer_logprob(inp, n_prompt, attention_mask=None):
    kw = dict(inp)
    if attention_mask is not None:
        kw["attention_mask"] = attention_mask
    logits = model(**kw).logits[0].float()
    lp = torch.log_softmax(logits[:-1], dim=-1)
    return float(lp.gather(-1, inp["input_ids"][0, 1:, None]).squeeze(-1)[n_prompt - 1:].sum())

# --- ground truth: load, or regenerate ---
if os.path.exists(CACHE):
    data = torch.load(CACHE, weights_only=False)
    print(f"loaded {len(data)} cached examples")
else:
    print("regenerating the LOO ground truth")
    types = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
    samples = []
    for t in types:
        for jp in sorted(glob.glob(os.path.join(DATA_ROOT, t, "*.json")))[:N_PER_TYPE]:
            m = json.load(open(jp)); ip = jp[:-5] + ".jpg"
            if os.path.exists(ip) and "gaze" in m and m.get("response"):
                samples.append(dict(type=t, img_path=ip, question=m["question"],
                                    answer=m["response"], gaze=m["gaze"]))
    data, t0 = [], time.time()
    for i, s in enumerate(samples):
        inp, n_prompt = build_inputs(S.load_image(s["img_path"]), s["question"], s["answer"])
        ids = inp["input_ids"][0].cpu()
        img_pos = torch.nonzero(ids == S._find_image_token_id(model, processor)).squeeze(-1)
        base = answer_logprob(inp, n_prompt)
        drops = torch.zeros(len(img_pos))
        for j in range(len(img_pos)):
            am = inp["attention_mask"].clone(); am[0, img_pos[j]] = 0
            drops[j] = base - answer_logprob(inp, n_prompt, am)
        data.append(dict(**s, base_logp=base, drops=drops))
        if (i + 1) % 5 == 0:
            print(f"  {i+1}/{len(samples)}  ({(time.time()-t0)/60:.1f} min)")
    torch.save(data, CACHE)

L_v = data[0]["drops"].numel(); G = int(round(math.sqrt(L_v))); N = len(data)

if os.path.exists(SINKF):
    sinks = torch.load(SINKF, weights_only=False)["sink_mask"].bool()
else:
    sc = []
    for d in data[:6]:
        o = S.make_smolvlm_output(image=S.load_image(d["img_path"]), question=d["question"])
        sc.append(VS.sink_scores(o.post_softmax, o.image_token_mask, o.text_token_mask,
                                 is_post_softmax=True)); del o
    score = VS.aggregate_sink_scores(sc); sinks = VS.detect_sinks(score)
    torch.save({"model": MODEL_ID, "L_v": L_v, "sink_mask": sinks, "sink_score": score}, SINKF)
    print(VS.sink_report(score, sinks))

print(f"{N} examples | L_v={L_v} ({G}x{G}) | sinks {int(sinks.sum())}")

## 2. Question-free predictors — one null-prompt forward per example

The same image, with a content-free prompt instead of the real question. We capture the merged
image-token embeddings (for the similarity and distinctiveness predictors) and the attention
(for null-prompt importance).

In [ ]:
def find_decoder_layers(model, n_expected=24):
    hits = [(n, m) for n, m in model.named_modules()
            if isinstance(m, torch.nn.ModuleList) and len(m) == n_expected]
    for n, m in hits:
        if any(t in n for t in ("text", "language", "llm")):
            return m
    return hits[0][1]

dec = find_decoder_layers(model)

def gaze_patch(gz):
    return min(G-1, int(gz["y_norm"]*G)) * G + min(G-1, int(gz["x_norm"]*G))

feat = []
for i, d in enumerate(data):
    img = S.load_image(d["img_path"])
    inp, _ = build_inputs(img, NULL_PROMPT)
    ids = inp["input_ids"][0].cpu()
    iid = S._find_image_token_id(model, processor)
    pad = tokenizer.pad_token_id
    im  = ids == iid
    tm  = (ids != iid) & (ids != (pad if pad is not None else -10**9))
    img_cols = torch.nonzero(im).squeeze(-1)

    store = {}
    def pre_hook(_m, args):
        store["h"] = args[0]
        return None
    hh = dec[0].register_forward_pre_hook(pre_hook)
    patched = S._patch_eager_globals(S._make_raw_capturing_eager(None))
    try:
        with torch.no_grad():
            model(**inp)
    finally:
        S._unpatch_eager_globals(patched); hh.remove()

    E = store["h"][0].detach().float().cpu()[img_cols]          # [L_v, d]
    raw = {}
    for m in model.modules():
        r = getattr(m, "_raw_attn_scores", None)
        if r is not None and r.shape[-1] == len(ids) and r.shape[-2] == len(ids):
            raw[int(getattr(m, "layer_idx", len(raw)))] = r[0].float()
        for a in ("_raw_attn_scores", "_post_attn"):
            if hasattr(m, a):
                delattr(m, a)

    maps, tp, _ = RS.sliced_maps_from_full(raw, im, tm)
    tt = tokenizer.convert_ids_to_tokens(ids[tp].tolist())
    cm = RS.content_text_mask(tt, tokenizer)
    if int(cm.sum()) == 0:
        cm = torch.ones(len(tt), dtype=torch.bool)
    null_imp, *_ = VS.image_importance(maps, cm, cand_mask=VS.candidate_mask(L_v, exclude=[sinks]))

    En = F.normalize(E, dim=-1)
    gp = gaze_patch(d["gaze"])
    feat.append(dict(gaze_sim=(En @ En[gp]),                       # untrained FRM (W = I)
                     distinct=1 - (En @ F.normalize(E.mean(0), dim=-1)),
                     tok_norm=E.norm(dim=-1),
                     null_imp=null_imp, gp=gp))
    del raw, maps, store, E, En
    if (i + 1) % 5 == 0:
        print(f"  {i+1}/{N}")
print("done")

## 3. Table 1 — all candidates (comparable with every earlier notebook)

In [ ]:
DX_SIG, DY_SIG, DX_MU, DY_MU = 2.69, 1.72, 0.95, -0.20     # fitted in the headroom notebook

def blob(gp, sc=DX_SIG, sr=DY_SIG, oc=DX_MU, orr=DY_MU):
    r0, c0 = divmod(gp, G); r0, c0 = r0 + orr, c0 + oc
    return torch.tensor([-(((i//G - r0)/sr)**2 + ((i%G - c0)/sc)**2) for i in range(L_v)])

def isotropic(gp):
    r0, c0 = divmod(gp, G)
    return torch.tensor([-math.hypot(i//G - r0, i % G - c0) for i in range(L_v)])

def z(v):
    v = v.float(); return (v - v.mean()) / v.std().clamp_min(1e-9)

g = torch.Generator().manual_seed(0)

def predictors(d, f):
    gp = f["gp"]
    return {
        "CTRL random":                  torch.rand(L_v, generator=g),
        "center (no gaze, no image)":   isotropic(L_v // 2),
        "gaze blob (geometry only)":    blob(gp),
        "token norm (image only)":      f["tok_norm"],
        "distinctiveness (image only)": f["distinct"],
        "null-prompt attn (image)":     f["null_imp"],
        "gaze-sim = untrained FRM":     f["gaze_sim"],
        "gaze-sim x blob":              z(f["gaze_sim"]) + z(blob(gp)),
    }

QUOTED = {"attn x grad  [sees question]": .350, "attention imp_a  [sees question]": .311}

def run_table(mask, title, quoted=None):
    cidx = torch.nonzero(mask, as_tuple=False).squeeze(-1)
    nc   = int(mask.sum())
    def topk(v, k):
        return set(cidx[torch.topk(v[cidx], k).indices].tolist())
    for k in (10,):
        chance, rows, used = k / nc, defaultdict(list), 0
        for d, f in zip(data, feat):
            if float(torch.topk(d["drops"][cidx], k).values[-1]) <= 1e-6:
                continue
            used += 1
            gt = topk(d["drops"], k)
            for n_, v in predictors(d, f).items():
                rows[n_].append(len(topk(v, k) & gt) / k)
        print(f"\n=== {title}")
        print(f"    precision@{k}   chance {chance:.1%}   candidates {nc}   usable {used}/{N}")
        print(f"    {'predictor':<32}{'prec':>8}{'x chance':>10}{'p':>8}")
        print("    " + "-" * 56)
        scored = {n_: float(np.mean(v)) for n_, v in rows.items()}
        if quoted:
            scored.update(quoted)
        for n_ in sorted(scored, key=lambda x: -scored[x]):
            if n_ in rows:
                a = np.array(rows[n_]); se = a.std(ddof=1)/max(np.sqrt(len(a)), 1e-9)
                p = 2*(1-norm.cdf(abs((a.mean()-chance)/se))) if se > 0 else 1.0
                tail = f"{p:>8.3f}{'  *' if p < 0.05 else ''}"
            else:
                tail = "   (quoted)"
            print(f"    {n_:<32}{scored[n_]:>8.1%}{scored[n_]/chance:>9.2f}x{tail}")
        return scored

t1 = run_table(VS.candidate_mask(L_v, exclude=[sinks]),
               "ALL CANDIDATES (sinks removed)", QUOTED)

## 4. Table 2 — fovea also removed: **FRM's actual job**

Spec section 3.1: the gaze cell and its 1-ring are already rendered at high resolution by Stage 2a,
so FRM never selects them. Scoring here asks the only question that matters for Stage 2b:
**how well can you find the relevant context you are NOT looking at?**

In [ ]:
def fovea_mask(gp):
    r0, c0 = divmod(gp, G)
    m = torch.zeros(L_v, dtype=torch.bool)
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            r, c = r0 + dr, c0 + dc
            if 0 <= r < G and 0 <= c < G:
                m[r * G + c] = True
    return m

# per-example fovea, so the candidate set differs per example -> score example-by-example
k = 10
rows, used, chances = defaultdict(list), 0, []
for d, f in zip(data, feat):
    mask = VS.candidate_mask(L_v, exclude=[sinks, fovea_mask(f["gp"])])
    cidx = torch.nonzero(mask, as_tuple=False).squeeze(-1)
    if float(torch.topk(d["drops"][cidx], k).values[-1]) <= 1e-6:
        continue
    used += 1; chances.append(k / int(mask.sum()))
    def topk(v):
        return set(cidx[torch.topk(v[cidx], k).indices].tolist())
    gt = topk(d["drops"])
    for n_, v in predictors(d, f).items():
        rows[n_].append(len(topk(v) & gt) / k)

chance = float(np.mean(chances))
print(f"=== FOVEA EXCLUDED - context you are NOT looking at")
print(f"    precision@{k}   chance {chance:.1%}   usable {used}/{N}")
print(f"    {'predictor':<32}{'prec':>8}{'x chance':>10}{'p':>8}")
print("    " + "-" * 56)
for n_ in sorted(rows, key=lambda x: -np.mean(rows[x])):
    a = np.array(rows[n_]); se = a.std(ddof=1)/max(np.sqrt(len(a)), 1e-9)
    p = 2*(1-norm.cdf(abs((a.mean()-chance)/se))) if se > 0 else 1.0
    print(f"    {n_:<32}{a.mean():>8.1%}{a.mean()/chance:>9.2f}x{p:>8.3f}"
          f"{'  *' if p < 0.05 else ''}")

print("\nNote: the teachers cannot be quoted here - their numbers were measured on the")
print("full candidate set. Comparable teacher numbers on this restricted set would need")
print("a re-run of the bake-off with the fovea excluded.")

## 5. Verdict — where does FRM's ceiling actually sit?

Read Table 2 (fovea excluded). `gaze-sim = untrained FRM` is the key row: it is the FRM
architecture with identity projections, so a trained FRM should land **above** it.

* **gaze-sim clearly above the gaze blob** -> the gaze token's embedding carries information about
  which distant tokens matter, over and above where the eye is. That is the FRM premise working at
  the representation level, before any training. **Build it.**
* **gaze-sim ~= gaze blob** -> the embedding adds nothing over pure geometry. A trained FRM would
  have to learn everything from scratch through `W_q, W_k`, and the spec's kill criterion is close
  to firing.
* **null-prompt attention or distinctiveness at the top** -> most of the recoverable signal is
  question-free *saliency*, which needs no gaze at all. Then the honest framing of the module is
  "learned saliency", not "gaze predicts context".
* **everything near chance once the fovea is removed** -> the far-context signal from Check 2 is not
  recoverable from gaze or image alone, only from the question. FRM cannot work, whatever the
  teacher.

Standing caveats: n=18-20, no paired tests, and the null prompt ("Describe the image.") is a
stand-in rather than a truly empty prompt. Treat gaps under ~5 points as unresolved.